In [11]:
!pip install -q deltalake pyarrow

In [12]:
import os
import shutil
import glob
import pandas as pd
from deltalake import DeltaTable, write_deltalake

print("Import Delta Lake berhasil.")

Import Delta Lake berhasil.


In [13]:
import os, sys, time, json, sqlite3, random, shutil, glob, subprocess
import numpy as np, pandas as pd

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "duckdb", "networkx", "shapely>=2.0", "tinydb", "deltalake>=1.0"], check=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/BigData"
else:
    BASE = os.path.abspath("./BigData")

DIR_P3 = f"{BASE}/Praktikum3"
DIR_SIMPAN = f"{BASE}/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"
os.makedirs(DIR_SIMPAN, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})
    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)
    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])
    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

DATA_TIRUAN = False
if not os.path.exists(TRIPS_BERSIH):
    print("!! trips_bersih/ tidak ditemukan -> memakai DATA TIRUAN.")
    TRIPS_BERSIH = bangkitkan_trips_sintetis(f"{DIR_SIMPAN}/trips_bersih_sintetis")
    DATA_TIRUAN = True
print("DATA_TIRUAN:", DATA_TIRUAN, "->", TRIPS_BERSIH)

!! trips_bersih/ tidak ditemukan -> memakai DATA TIRUAN.
DATA_TIRUAN: True -> /content/drive/MyDrive/BigData/Praktikum4/trips_bersih_sintetis


In [15]:
# Membaca dataframe dari data Parquet yang telah disiapkan
df = pd.read_parquet(TRIPS_BERSIH)

# Menyelaraskan nama kolom data tiruan dengan kolom delta yang dibutuhkan
# Jika nama kolom aslinya berbeda (misal dari data sintetis), kita lakukan mapping
mapping_kolom = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "nama_pembayaran": "metode_bayar"
}

df_mapped = df.rename(columns=mapping_kolom)

KOLOM_DELTA = [
    "waktu_jemput",
    "zona_jemput",
    "zona_tujuan",
    "wilayah_jemput",
    "jarak_mil",
    "total_bayar",
    "metode_bayar"
]

data_delta = (
    df_mapped[KOLOM_DELTA]
    .sample(n=200_000, random_state=42, replace=True)
    .reset_index(drop=True)
)

print("Jumlah data:", len(data_delta))
print(data_delta.head())

Jumlah data: 200000
         waktu_jemput  zona_jemput  zona_tujuan wilayah_jemput  jarak_mil  \
0 2023-01-26 08:44:00          166          215       Brooklyn       1.04   
1 2023-01-14 23:01:00          247          144         Queens       4.12   
2 2023-01-03 02:52:00          144          109       Brooklyn       2.03   
3 2023-01-09 09:10:00          221          217      Manhattan       3.63   
4 2023-01-30 12:57:00           67          179       Brooklyn       6.28   

   total_bayar  metode_bayar  
0        13.45         Tunai  
1        19.76  Kartu kredit  
2        19.28  Kartu kredit  
3        20.85         Tunai  
4        23.85  Kartu kredit  


In [16]:
PATH_DELTA = "/content/trips_delta"

if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

print("Path Delta:", PATH_DELTA)

Path Delta: /content/trips_delta


In [17]:
hari1 = data_delta.iloc[:100_000].copy()

write_deltalake(
    PATH_DELTA,
    hari1,
    mode="overwrite"
)

dt = DeltaTable(PATH_DELTA)

print("=== HARI 1 ===")
print("Version :", dt.version())
print("Baris   :", len(dt.to_pandas()))

assert dt.version() == 0
assert len(dt.to_pandas()) == 100_000

=== HARI 1 ===
Version : 0
Baris   : 100000


In [18]:
hari2_tambah = data_delta.iloc[100_000:150_000].copy()

write_deltalake(
    PATH_DELTA,
    hari2_tambah,
    mode="append"
)

dt = DeltaTable(PATH_DELTA)

print("=== HARI 2 - SETELAH TAMBAH DATA ===")
print("Version :", dt.version())
print("Baris   :", len(dt.to_pandas()))

assert dt.version() == 1
assert len(dt.to_pandas()) == 150_000

=== HARI 2 - SETELAH TAMBAH DATA ===
Version : 1
Baris   : 150000


In [19]:
dt = DeltaTable(PATH_DELTA)

data_sebelum_koreksi = dt.to_pandas()

mask_koreksi = (
    (data_sebelum_koreksi["wilayah_jemput"] == "Bronx") &
    (data_sebelum_koreksi["metode_bayar"] == "Tunai")
)

n_koreksi = int(mask_koreksi.sum())

print("Jumlah baris yang memenuhi kondisi koreksi:", n_koreksi)

dt.update(
    predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
    updates={
        "total_bayar": "total_bayar + 1.0"
    }
)

dt = DeltaTable(PATH_DELTA)

print("\n=== HARI 2 - SETELAH KOREKSI ===")
print("Version :", dt.version())
print("Baris   :", len(dt.to_pandas()))

Jumlah baris yang memenuhi kondisi koreksi: 2518

=== HARI 2 - SETELAH KOREKSI ===
Version : 2
Baris   : 150000


In [20]:
# Versi 0 = keadaan akhir Hari 1
v0 = DeltaTable(
    PATH_DELTA,
    version=0
).to_pandas()

# Versi 1 = setelah penambahan Hari 2
v1 = DeltaTable(
    PATH_DELTA,
    version=1
).to_pandas()

# Versi terbaru = setelah koreksi
v2 = DeltaTable(
    PATH_DELTA
).to_pandas()

print("=== TIME TRAVEL ===")
print("Versi 0 :", len(v0), "baris")
print("Versi 1 :", len(v1), "baris")
print("Versi 2 :", len(v2), "baris")

assert len(v0) == 100_000
assert len(v1) == 150_000
assert len(v2) == 150_000

=== TIME TRAVEL ===
Versi 0 : 100000 baris
Versi 1 : 150000 baris
Versi 2 : 150000 baris


In [21]:
selisih_total = (
    v2["total_bayar"].sum()
    - v1["total_bayar"].sum()
)

print("Jumlah baris dikoreksi :", n_koreksi)
print("Selisih total_bayar    :", round(selisih_total, 2))

assert round(selisih_total, 2) == round(n_koreksi * 1.0, 2)

Jumlah baris dikoreksi : 2518
Selisih total_bayar    : 2518.0


In [22]:
dt = DeltaTable(PATH_DELTA)

history = dt.history()

print("=== HISTORY ===")

for h in history:
    print(
        "version =", h.get("version"),
        "| operation =", h.get("operation")
    )

=== HISTORY ===
version = 2 | operation = UPDATE
version = 1 | operation = WRITE
version = 0 | operation = WRITE


In [23]:
LOG_PATH = os.path.join(
    PATH_DELTA,
    "_delta_log"
)

log_files = sorted(
    os.listdir(LOG_PATH)
)

print("Jumlah file log:", len(log_files))

for nama_file in log_files:

    path_file = os.path.join(
        LOG_PATH,
        nama_file
    )

    print("\n" + "=" * 70)
    print(nama_file)
    print("=" * 70)

    with open(
        path_file,
        "r",
        encoding="utf-8"
    ) as f:

        print(f.read())

Jumlah file log: 3

00000000000000000000.json
{"commitInfo":{"timestamp":1791204236004,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":100000,"execution_time_ms":62,"num_retries":0},"clientVersion":"delta-rs.py-1.6.6"}}
{"protocol":{"minReaderVersion":3,"minWriterVersion":7,"readerFeatures":["timestampNtz"],"writerFeatures":["timestampNtz"]}}
{"metaData":{"id":"3911d39e-4e71-454b-b25b-57f150a1dcaf","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"waktu_jemput\",\"type\":\"timestamp_ntz\",\"nullable\":true,\"metadata\":{}},{\"name\":\"zona_jemput\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"zona_tujuan\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"wilayah_jemput\",\"type\":\"string\",\"nullable\":true,\"metadata\":

In [24]:

OUTPUT_LOG = os.path.join(
    DIR_SIMPAN,
    "delta_log"
)

if os.path.exists(OUTPUT_LOG):
    shutil.rmtree(OUTPUT_LOG)

shutil.copytree(
    LOG_PATH,
    OUTPUT_LOG
)

print(
    "Folder log berhasil disalin ke:",
    OUTPUT_LOG
)

Folder log berhasil disalin ke: /content/drive/MyDrive/BigData/Praktikum4/delta_log


In [25]:
ZIP_LOG = shutil.make_archive(
    os.path.join(DIR_SIMPAN, "delta_log"),
    "zip",
    OUTPUT_LOG
)

print("ZIP berhasil dibuat:")
print(ZIP_LOG)

ZIP berhasil dibuat:
/content/drive/MyDrive/BigData/Praktikum4/delta_log.zip


In [26]:
snapshot_path = os.path.join(
    DIR_SIMPAN,
    "delta_day1_snapshot.csv"
)

v0.to_csv(
    snapshot_path,
    index=False
)

print(
    "Snapshot Hari 1:",
    snapshot_path
)

Snapshot Hari 1: /content/drive/MyDrive/BigData/Praktikum4/delta_day1_snapshot.csv
